Start by importing the required packages.

In [2]:
import re
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

Load in a CSV file and look and the column names and datatypes.

In [4]:
atp2023 = pd.read_csv('atp_matches_2023.csv')
atp2023.dtypes

tourney_id             object
tourney_name           object
surface                object
draw_size               int64
tourney_level          object
tourney_date            int64
match_num               int64
winner_id               int64
winner_seed           float64
winner_entry           object
winner_name            object
winner_hand            object
winner_ht             float64
winner_ioc             object
winner_age            float64
loser_id                int64
loser_seed            float64
loser_entry            object
loser_name             object
loser_hand             object
loser_ht              float64
loser_ioc              object
loser_age             float64
score                  object
best_of                 int64
round                  object
minutes               float64
w_ace                 float64
w_df                  float64
w_svpt                float64
w_1stIn               float64
w_1stWon              float64
w_2ndWon              float64
w_SvGms   

Drop unnecessary columns, change the date column to datetime, and list unique values in the tourney_name column.

In [6]:
atp2023 = atp2023[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]
atp2023['tourney_date'] = pd.to_datetime(atp2023['tourney_date'].astype(str), format='%Y%m%d')
atp2023.dtypes

tourney_id               object
tourney_name             object
surface                  object
draw_size                 int64
tourney_level            object
tourney_date     datetime64[ns]
dtype: object

In [7]:
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals', 'Davis Cup

Remove the values that contain "Davis Cup". These won't be used in the analysis.

In [9]:
atp2023 = atp2023[~atp2023['tourney_name'].str.contains('Davis Cup', case=False, na=False)]
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=obj

Remove "Masters" from any of the values. This will help find the locations.

In [11]:
atp2023["tourney_name"] = atp2023["tourney_name"].str.replace("masters", "", case=False, regex=True).str.strip()
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells', 'Miami', 'Estoril', 'Houston', 'Marrakech',
       'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka', 'Madrid',
       'Rome', 'Geneva', 'Lyon', 'Roland Garros', 's Hertogenbosch',
       'Stuttgart', 'Halle', "Queen's Club", 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos', 'Washington', 'Canada',
       'Cincinnati', 'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai',
       'Astana', 'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=object)

Change the tournament names without clear locations to be more specific.

In [13]:
atp2023["tourney_name"] = atp2023["tourney_name"].replace({
    "Australian Open": "Melbourne",
    "Roland Garros": "Paris",
    "Queen's Club": "London",
    "Canada": "Toronto",
    "Us Open": "New York",
    "Tour Finals": "Turin",
    "NextGen Finals": "Jeddah",
    "Cordoba": "Cordoba, Argentina",
    "Los Cabos": "Los Cabos, Baja California Sur, Mexico",
    "Santiago": "Santiago, Chile",
})
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Melbourne', 'Cordoba, Argentina', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai',
       'Santiago, Chile', 'Indian Wells', 'Miami', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka',
       'Madrid', 'Rome', 'Geneva', 'Lyon', 'Paris', 's Hertogenbosch',
       'Stuttgart', 'Halle', 'London', 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos, Baja California Sur, Mexico',
       'Washington', 'Toronto', 'Cincinnati', 'Winston-Salem', 'New York',
       'Chengdu', 'Zhuhai', 'Astana', 'Beijing', 'Shanghai', 'Stockholm',
       'Tokyo', 'Antwerp', 'Basel', 'Vienna', 'Metz', 'Sofia', 'Turin',
       'Jeddah'], dtype=object)

Change "United Cup" to the locations where the event is held.

In [15]:
atp2023["tourney_name"] = atp2023["tourney_name"].apply(
    lambda x: ["Brisbane", "Sydney", "Perth"] if x == "United Cup" else [x]
)

atp2023 = atp2023.explode("tourney_name")
atp2023["tourney_name"].unique()

array(['Brisbane', 'Sydney', 'Perth', 'Adelaide 1', 'Pune', 'Auckland',
       'Adelaide 2', 'Melbourne', 'Cordoba, Argentina', 'Dallas',
       'Montpellier', 'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai',
       'Santiago, Chile', 'Indian Wells', 'Miami', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka',
       'Madrid', 'Rome', 'Geneva', 'Lyon', 'Paris', 's Hertogenbosch',
       'Stuttgart', 'Halle', 'London', 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos, Baja California Sur, Mexico',
       'Washington', 'Toronto', 'Cincinnati', 'Winston-Salem', 'New York',
       'Chengdu', 'Zhuhai', 'Astana', 'Beijing', 'Shanghai', 'Stockholm',
       'Tokyo', 'Antwerp', 'Basel', 'Vienna', 'Metz', 'Sofia', 'Turin',
       'Jeddah'], dtype=object)

Drop duplicate tournament names so you have 1 row for each unique location

In [17]:
atp2023 = atp2023.drop_duplicates(subset=["tourney_id", "tourney_name"]).reset_index(drop=True)
atp2023

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date
0,2023-9900,Brisbane,Hard,18,A,2023-01-02
1,2023-9900,Sydney,Hard,18,A,2023-01-02
2,2023-9900,Perth,Hard,18,A,2023-01-02
3,2023-2843,Adelaide 1,Hard,32,A,2023-01-02
4,2023-0891,Pune,Hard,32,A,2023-01-02
...,...,...,...,...,...,...
64,2023-0352,Paris,Hard,64,M,2023-10-30
65,2023-0341,Metz,Hard,32,A,2023-11-06
66,2023-7434,Sofia,Hard,32,A,2023-11-06
67,2023-0605,Turin,Hard,8,A,2023-11-13


Create a dataframe that contains a list of the different locations tournaments are held and the locations for each one.

In [19]:
geolocator = Nominatim(user_agent="ATP_Tour_2023")                    # The file the function is being run on
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)        # Nominatim rate limit

unique_cities = atp2023["tourney_name"]                               # Create a list of all the cities

location_data = {}                                                    # Create dictionary
for city in unique_cities:                                            # Run for loop with geocode function to locate each city 
    location = geocode(city)
    if location:                                                      # If else logic to find latitude and longitude of each city and place in dictionary
        location_data[city] = (location.latitude, location.longitude)
    else:
        print(f"Could not geocode: {city}")                           # Flag any misses
location_data
coord_df = pd.DataFrame.from_dict(                                    # Create df from dictionary populated above 
    location_data, orient="index", columns=["latitude", "longitude"]
)
coord_df.index.name = "tourney_name"                                  # Make the index the tournament name/city and reset the index
coord_df = coord_df.reset_index()
coord_df

,tourney_name,latitude,longitude
0,Brisbane,-27.468962,153.023501
1,Sydney,-33.869844,151.208285
2,Perth,-31.955897,115.860578
3,Adelaide 1,-34.819459,138.504749
4,Pune,18.521374,73.854507
...,...,...,...
63,Vienna,48.208354,16.372504
64,Metz,49.119696,6.176355
65,Sofia,42.697703,23.321736
66,Turin,45.067755,7.682489


In [20]:
atp2023 = atp2023.merge(coord_df, on="tourney_name", how="left")
atp2023

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,latitude,longitude
0,2023-9900,Brisbane,Hard,18,A,2023-01-02,-27.468962,153.023501
1,2023-9900,Sydney,Hard,18,A,2023-01-02,-33.869844,151.208285
2,2023-9900,Perth,Hard,18,A,2023-01-02,-31.955897,115.860578
3,2023-2843,Adelaide 1,Hard,32,A,2023-01-02,-34.819459,138.504749
4,2023-0891,Pune,Hard,32,A,2023-01-02,18.521374,73.854507
...,...,...,...,...,...,...,...,...
64,2023-0352,Paris,Hard,64,M,2023-10-30,48.858890,2.320041
65,2023-0341,Metz,Hard,32,A,2023-11-06,49.119696,6.176355
66,2023-7434,Sofia,Hard,32,A,2023-11-06,42.697703,23.321736
67,2023-0605,Turin,Hard,8,A,2023-11-13,45.067755,7.682489


Fix a data error for the level of the Turin event

In [22]:
atp2023.loc[atp2023['tourney_name'] == 'Turin', 'tourney_level'] = 'F'

Make the data in the tourney_level column clearer

In [24]:
atp2023["tourney_level"] = atp2023["tourney_level"].replace({
    "A": "Tour Event",
    "G": "Grand Slam",
    "M": "Masters",
    "F": "Finals"
})

Fix up the column names

In [50]:
atp2023 = atp2023.rename(columns={
    "tourney_id": "Tournament_ID",
    "tourney_name": "Tournament_City",
    "surface": "Surface",
    "draw_size": "Draw_Size",
    "tourney_level": "Tournament_Level",
    "tourney_date": "Tournament_Date",
    "latitude": "Latitude",
    "longitude": "Longitude"
})

In [51]:
atp2023

,Tournament_ID,Tournament_City,Surface,Draw_Size,Tournament_Level,Tournament_Date,Latitude,Longitude
0,2023-9900,Brisbane,Hard,18,Tour Event,2023-01-02,-27.468962,153.023501
1,2023-9900,Sydney,Hard,18,Tour Event,2023-01-02,-33.869844,151.208285
2,2023-9900,Perth,Hard,18,Tour Event,2023-01-02,-31.955897,115.860578
3,2023-2843,Adelaide 1,Hard,32,Tour Event,2023-01-02,-34.819459,138.504749
4,2023-0891,Pune,Hard,32,Tour Event,2023-01-02,18.521374,73.854507
...,...,...,...,...,...,...,...,...
64,2023-0352,Paris,Hard,64,Masters,2023-10-30,48.858890,2.320041
65,2023-0341,Metz,Hard,32,Tour Event,2023-11-06,49.119696,6.176355
66,2023-7434,Sofia,Hard,32,Tour Event,2023-11-06,42.697703,23.321736
67,2023-0605,Turin,Hard,8,Finals,2023-11-13,45.067755,7.682489


In [52]:
atp2023.to_csv("atp_tour_2023.csv", index=False)

In [53]:
atp2023.dtypes

Tournament_ID               object
Tournament_City             object
Surface                     object
Draw_Size                    int64
Tournament_Level            object
Tournament_Date     datetime64[ns]
Latitude                   float64
Longitude                  float64
dtype: object